# Module 1 — Symbolic AI
## Build, test, and break a tiny expert system

**Estimated time:** 90–120 minutes  \n**Level:** beginner to intermediate Python

In this notebook, you will build a very small rule-based configuration adviser for a fictional computer lab. It is inspired by historical expert systems such as XCON: people explicitly describe facts and write rules, then the system applies the rules.

You will see that this approach is transparent and useful for anticipated cases, but needs extra human-authored rules when information is missing, contradictory, noisy, or new. That limitation motivates Module 2: learning from data.

## Learning outcomes

After this notebook, you can:

1. Represent a simple situation as explicit facts.
2. Describe knowledge using IF–THEN rules.
3. Run a small rule evaluator and read its explanation trace.
4. Identify missing, noisy, contradictory, and unseen input as limitations of a small rule-based system.
5. Explain why learning patterns from examples can become attractive.


## 1. Environment check

This notebook uses only standard Python. No installation and no external data are required. Run the next cell.

In [ ]:
import sys

print(f'Python version: {sys.version.split()[0]}')
print('Environment check passed: standard Python is enough for this notebook.')

## 2. Facts: what the system is told

A symbolic system does not see a real computer lab. It sees only the symbols that people provide. Here, a request is represented by a Python dictionary. Each key is a fact name and each value is the stated value of that fact.

Read the request below, then run the cell.

In [ ]:
normal_request = {
    'workload': 'visualisation',
    'needs_gpu': True,
    'memory_gb': 32,
    'operating_system': 'linux',
}

for fact_name, fact_value in normal_request.items():
    print(f'- {fact_name}: {fact_value}')

### Checkpoint

The dictionary is the system's small, explicit model of this request. If a useful fact is absent, the system cannot silently recover it from common sense.

## 3. Rules: knowledge written by people

Each rule has:

- a name, so we can explain what happened;
- conditions, written as fact/value pairs; and
- a conclusion, which is the recommended workstation type.

The first rule says:

> IF the workload is visualisation AND a GPU is needed, THEN recommend a graphics workstation.


In [ ]:
rules = [
    {
        'name': 'visualisation_needs_graphics',
        'conditions': {
            'workload': 'visualisation',
            'needs_gpu': True,
        },
        'conclusion': 'graphics_workstation',
    },
    {
        'name': 'office_work_uses_standard_machine',
        'conditions': {
            'workload': 'office',
            'needs_gpu': False,
        },
        'conclusion': 'standard_workstation',
    },
]

for rule in rules:
    print(f"{rule['name']}: IF {rule['conditions']} THEN {rule['conclusion']}")

## 4. A small rule evaluator

The next cell contains a completed evaluator. Read it at a high level before running it.

It checks every rule, records rules whose conditions match the facts, and produces both a decision and an explanation trace. A real expert system can be much more complex; this small version is deliberately transparent.

In [ ]:
def rule_matches(facts, rule):
    """Return True only when every rule condition matches a supplied fact."""
    return all(
        facts.get(fact_name) == expected_value
        for fact_name, expected_value in rule['conditions'].items()
    )


def evaluate_rules(facts, rules):
    """Evaluate all rules and return a decision plus a human-readable trace."""
    matched_rules = [rule for rule in rules if rule_matches(facts, rule)]
    conclusions = sorted({rule['conclusion'] for rule in matched_rules})

    if len(conclusions) == 1:
        decision = conclusions[0]
    elif len(conclusions) > 1:
        decision = 'manual_review'
    elif 'needs_gpu' not in facts:
        decision = 'insufficient_information'
    else:
        decision = 'unsupported_request'

    trace = []
    for rule in matched_rules:
        trace.append({
            'rule': rule['name'],
            'conditions': rule['conditions'],
            'conclusion': rule['conclusion'],
        })

    return {
        'decision': decision,
        'matched_rules': trace,
    }


def show_result(result):
    print(f"Decision: {result['decision']}")
    if result['matched_rules']:
        print('Applied rule trace:')
        for item in result['matched_rules']:
            print(f"- {item['rule']} → {item['conclusion']}")
    else:
        print('Applied rule trace: no rule matched.')

## 5. Normal case: transparency in action

Run the evaluator on the visualisation request. You should see one recommendation and the name of the rule that produced it.

In [ ]:
normal_result = evaluate_rules(normal_request, rules)
show_result(normal_result)

assert normal_result['decision'] == 'graphics_workstation'
print('\nSelf-check passed: the expected normal-case decision was produced.')

### Why this is a strength

The result is predictable and auditable. We can answer: **Why did the system decide this?** We can inspect the facts and the exact rule that fired. This transparency was one important attraction of symbolic expert systems.

## 6. Extend the rule base

A lab now asks for a machine for data analysis. The human designer must convert this requirement into an explicit rule.

The example below adds a rule. Read it, run it, and then change one condition to see how the behaviour changes.

**Guided task:** Change the memory threshold from `32` to `16`, run the cell again, and consider whether that change would be safe for every future request.

In [ ]:
# TODO (guided): experiment by changing the required memory value below.
data_analysis_rule = {
    'name': 'data_analysis_needs_memory',
    'conditions': {
        'workload': 'data_analysis',
        'memory_gb': 32,
    },
    'conclusion': 'data_analysis_workstation',
}

if data_analysis_rule not in rules:
    rules.append(data_analysis_rule)

data_analysis_request = {
    'workload': 'data_analysis',
    'needs_gpu': False,
    'memory_gb': 32,
    'operating_system': 'linux',
}

data_analysis_result = evaluate_rules(data_analysis_request, rules)
show_result(data_analysis_result)
assert data_analysis_result['decision'] == 'data_analysis_workstation'

### Reflection: knowledge acquisition

Before the system could make this new recommendation, somebody had to decide:

- which facts matter;
- what values count as sufficient;
- what conclusion is appropriate; and
- how to test that the new rule does not interfere with older rules.

This is a miniature version of the **knowledge-acquisition bottleneck**.

## 7. Missing information

Now remove the `needs_gpu` fact. The evaluator has an explicit fallback for this one missing fact. Notice that this fallback is itself another rule-like decision created by a human.

In [ ]:
missing_information_request = {
    'workload': 'visualisation',
    'memory_gb': 32,
    'operating_system': 'linux',
}

missing_result = evaluate_rules(missing_information_request, rules)
show_result(missing_result)
assert missing_result['decision'] == 'insufficient_information'

### Checkpoint

The system did not infer whether a GPU was needed. It returned `insufficient_information` only because its designer explicitly chose that fallback behaviour.

## 8. Noisy input

Real input is often inconsistent. Here, the workload has an extra space and different capitalisation. The rule expects exactly `visualisation`, so no intended rule matches.

In [ ]:
noisy_request = {
    'workload': 'Visualisation ',
    'needs_gpu': True,
    'memory_gb': 32,
    'operating_system': 'linux',
}

noisy_result = evaluate_rules(noisy_request, rules)
show_result(noisy_result)
assert noisy_result['decision'] == 'unsupported_request'

**Optional guided improvement:** A programmer could add input normalisation, for example by converting text to lower case and removing surrounding spaces. That improves this particular problem, but it is another requirement that must be anticipated, implemented, and tested.

## 9. Contradictory facts and conflicting conclusions

Suppose one source says a GPU is required while another says a GPU is not allowed. We add a rule for the second fact. Both rules can then match the same request.

Our simple policy is: when multiple incompatible conclusions apply, return `manual_review`. This is not automatic intelligence; it is a conflict policy that people designed.

In [ ]:
gpu_forbidden_rule = {
    'name': 'gpu_not_allowed_uses_standard_machine',
    'conditions': {
        'gpu_not_allowed': True,
    },
    'conclusion': 'standard_workstation',
}

if gpu_forbidden_rule not in rules:
    rules.append(gpu_forbidden_rule)

contradictory_request = {
    'workload': 'visualisation',
    'needs_gpu': True,
    'gpu_not_allowed': True,
    'memory_gb': 32,
    'operating_system': 'linux',
}

conflict_result = evaluate_rules(contradictory_request, rules)
show_result(conflict_result)
assert conflict_result['decision'] == 'manual_review'

## 10. An unseen request

The system now receives a workload it has never been taught: `quantum_simulation`. It may resemble data analysis to a person, but the rule base has no explicit rule connecting these concepts.

In [ ]:
unseen_request = {
    'workload': 'quantum_simulation',
    'needs_gpu': True,
    'memory_gb': 64,
    'operating_system': 'linux',
}

unseen_result = evaluate_rules(unseen_request, rules)
show_result(unseen_result)
assert unseen_result['decision'] == 'unsupported_request'

## 11. Final self-check

Run the following cell. It checks the high-level behaviour of the cases explored in this notebook.

In [ ]:
expected_decisions = {
    'normal': 'graphics_workstation',
    'data_analysis': 'data_analysis_workstation',
    'missing_information': 'insufficient_information',
    'noisy': 'unsupported_request',
    'contradictory': 'manual_review',
    'unseen': 'unsupported_request',
}

actual_decisions = {
    'normal': normal_result['decision'],
    'data_analysis': data_analysis_result['decision'],
    'missing_information': missing_result['decision'],
    'noisy': noisy_result['decision'],
    'contradictory': conflict_result['decision'],
    'unseen': unseen_result['decision'],
}

assert actual_decisions == expected_decisions
print('All Module 1 self-checks passed.')

## 12. Reflection: why move toward learning from data?

Write short answers below, individually or with a partner.

1. What did this rule-based system do well?
2. Which cases required a special rule, fallback, or policy?
3. What would become difficult if the lab supported hundreds of workloads, hardware options, and changing requirements?
4. Imagine that you had thousands of historical requests and successful configurations. Why might learning patterns from examples become attractive?

### Takeaway

Symbolic AI can be transparent, controllable, and effective in a narrow, stable domain. But people must explicitly acquire, encode, test, and maintain the rules. Module 2 introduces a different approach: learning useful patterns from data rather than writing every rule by hand.